# Qwen3.8-27B Uncensored FP8 on Kaggle TPU v5e-8

This notebook downloads and runs `orcarouter/Qwen3.8-27B-Uncensored-FP8` through the FP8 wrapper in this repo.

**Before running:** set Accelerator to **TPU VM v5e-8**, enable **Internet**, accept the model's Hugging Face access conditions, and add a Kaggle secret named **HF_TOKEN**.


In [ ]:
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('HF_TOKEN')
if not token:
    raise RuntimeError('Missing Kaggle secret HF_TOKEN')
os.environ['HF_TOKEN'] = token
print('HF_TOKEN loaded from Kaggle Secrets (value not printed).')


## Download the serving scripts

The notebook fetches the two Python entrypoints from this repository so the large embedded MTP patch remains maintained in one place.


In [ ]:
import urllib.request

base = 'https://raw.githubusercontent.com/phakoda/kaggle-tpu-lab/main'
for rel in ['kernel/serve_qwen38.py', 'kernel/serve_qwen38_fp8.py']:
    out = Path('/kaggle/working') / Path(rel).name
    urllib.request.urlretrieve(f'{base}/{rel}', out)
    print('downloaded', out)


## Configuration

The default keeps native 262K context and 4 sequences. For the first validation run, `text_only=True` is a useful way to avoid vision-tower compilation.


In [ ]:
import json

config = {
    'max_model_len': 262144,
    'max_num_seqs': 4,
    'mtp_tokens': 3,
    'text_only': False,
    'keepalive_min': 480,
    'served_model_name': 'qwen3.8-27b-uncensored-fp8',
}
Path('/kaggle/working/serve_config.json').write_text(json.dumps(config, indent=2))
config


## Start the server

This cell stays active while the endpoint is serving. The output prints the Cloudflare URL, API key, model name, health status, and a short decode benchmark.


In [ ]:
%cd /kaggle/working
%run serve_qwen38_fp8.py
